In [1]:
# -----------------------------
# Cell 1: Imports and paths
# -----------------------------

import os
import re
import json
import time
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm import tqdm

try:
    from google import genai
except ImportError:
    raise ImportError("Please install google-genai first: pip install google-genai")

OUTPUT_DIR = Path("region_sensitivity_outputs")
EVAL_OUTPUT_DIR = Path("region_sensitivity_eval_outputs")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_1800_eval.json"
DEBUG_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_debug_15.json"

print("Official QA path:", OFFICIAL_QA_PATH)
print("Debug QA path:", DEBUG_QA_PATH)
print("Eval output dir:", EVAL_OUTPUT_DIR)


Official QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Debug QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_debug_15.json
Eval output dir: region_sensitivity_eval_outputs


In [10]:
# -----------------------------
# Cell 2: Load QA data
# -----------------------------
# Use True for testing 15 examples.
# Change to False when running the official 1800-example evaluation.

USE_DEBUG = False

QA_PATH = DEBUG_QA_PATH if USE_DEBUG else OFFICIAL_QA_PATH

with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

eval_df = pd.DataFrame(qa_data)

print("Loaded QA:", eval_df.shape)
print(eval_df["task_type"].value_counts())
print()
print("Answer distribution by task:")
print(pd.crosstab(eval_df["task_type"], eval_df["answer"]))
print()
print(eval_df[["id", "task_type", "answer", "gold_region"]].head())


Loaded QA: (1800, 9)
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64

Answer distribution by task:
answer                  A    B    C    D
task_type                               
management_priority   162  157  150  131
pairwise_comparison   293  307    0    0
top_sensitive_region  161  144  142  153

                                       id            task_type answer  \
0  rs_context_v2_pairwise_comparison_0001  pairwise_comparison      B   
1  rs_context_v2_pairwise_comparison_0002  pairwise_comparison      B   
2  rs_context_v2_pairwise_comparison_0003  pairwise_comparison      B   
3  rs_context_v2_pairwise_comparison_0004  pairwise_comparison      B   
4  rs_context_v2_pairwise_comparison_0005  pairwise_comparison      B   

  gold_region  
0  Leichhardt  
1   Warringah  
2  Parramatta  
3  Parramatta  
4   Warringah  


In [3]:
# -----------------------------
# Cell 3: Check evaluation fields
# -----------------------------

required_cols = [
    "id",
    "task_type",
    "system_prompt",
    "question",
    "answer",
    "gold_region"
]

missing_cols = [col for col in required_cols if col not in eval_df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("All required columns found.")
print("Columns in eval_df:")
print(eval_df.columns.tolist())


All required columns found.
Columns in eval_df:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']


In [4]:
# -----------------------------
# Cell 4: Extract option letter from model response
# -----------------------------

def extract_option_letter(response_text):
    """
    Extract A/B/C/D from model output.

    This function avoids taking the first option letter mentioned in the reasoning.
    It prioritises final-answer patterns such as:
    - "The final answer is C"
    - "Answer: C"
    - "\\boxed{C}"
    """

    if response_text is None:
        return None

    text = str(response_text).strip()
    upper_text = text.upper().strip()

    # Direct one-letter answer, allowing punctuation
    direct_match = re.match(r"^\s*([ABCD])\s*[\.\)]?\s*$", upper_text)
    if direct_match:
        return direct_match.group(1)

    # Normalise whitespace
    upper_text = re.sub(r"\s+", " ", upper_text)

    # LaTeX boxed answer, e.g. \boxed{C}
    boxed_match = re.search(r"\\?BOXED\{([ABCD])\}", upper_text)
    if boxed_match:
        return boxed_match.group(1)

    # Common final-answer patterns
    final_patterns = [
        r"THE FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER\s*[:\-]?\s*([ABCD])",
        r"THE ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER\s*[:\-]?\s*([ABCD])",
        r"OPTION\s*([ABCD])",
        r"CHOOSE\s*([ABCD])",
    ]

    for pattern in final_patterns:
        matches = re.findall(pattern, upper_text)
        if matches:
            return matches[-1]

    # Fallback: use the last standalone A/B/C/D, not the first.
    # This handles outputs that discuss all options before giving a final answer.
    all_matches = re.findall(r"\b([ABCD])\b", upper_text)

    if all_matches:
        return all_matches[-1]

    return None


# Quick test
test_outputs = [
    "A",
    "A.",
    "Option B",
    "The answer is C.",
    "D because ...",
    "The final answer is \\boxed{C}",
    "After comparing A, B, C, and D, the final answer is C.",
    "unknown"
]

for x in test_outputs:
    print(x, "->", extract_option_letter(x))


A -> A
A. -> A
Option B -> B
The answer is C. -> C
D because ... -> D
The final answer is \boxed{C} -> C
After comparing A, B, C, and D, the final answer is C. -> C
unknown -> None


In [5]:
# -----------------------------
# Cell 5: Set Gemini API key and create client
# -----------------------------
# Do not paste your API key directly into the notebook.
# This will ask for the key securely.

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini client ready.")


Enter your Gemini API key:  ········


Gemini client ready.


In [6]:
# -----------------------------
# Cell 6: Gemini API call function
# -----------------------------

def call_gemini_model(system_prompt, question, model_name="gemini-2.5-flash"):
    """
    Call Gemini model and return raw text response.
    Only system_prompt and question are sent to the model.
    Gold answer fields are not sent.
    """

    full_prompt = (
        system_prompt
        + "\n\n"
        + question
        + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
        + "Do not explain your reasoning."
    )

    response = gemini_client.models.generate_content(
        model=model_name,
        contents=full_prompt
    )

    return response.text


In [18]:
def call_gemini_model_with_retry(
    system_prompt,
    question,
    model_name="gemini-2.5-flash",
    max_retries=8,
    base_sleep=20
):
    """
    Call Gemini with retry logic for temporary API errors.
    Do not retry when prepaid credits are depleted.
    """

    retry_keywords = [
        "503",
        "UNAVAILABLE",
        "500",
        "INTERNAL",
        "DEADLINE_EXCEEDED"
    ]

    for attempt in range(max_retries):
        try:
            return call_gemini_model(
                system_prompt=system_prompt,
                question=question,
                model_name=model_name
            )

        except Exception as e:
            error_text = str(e)

            # Do not retry if the balance is depleted
            if "prepayment credits are depleted" in error_text.lower():
                raise RuntimeError(
                    "Google Gemini prepaid credits are depleted. "
                    "Please top up in AI Studio before continuing."
                )

            # Retry only temporary service errors
            should_retry = any(
                keyword in error_text
                for keyword in retry_keywords
            )

            if should_retry:
                wait_time = base_sleep * (attempt + 1)
                print(f"Temporary API error. Waiting {wait_time} seconds before retry...")
                print("Error:", error_text[:300])
                time.sleep(wait_time)
            else:
                raise e

    raise RuntimeError("Max retries exceeded due to repeated temporary API errors.")

In [8]:
# -----------------------------
# Cell 8: Test one Gemini call
# -----------------------------

TEST_MODEL_NAME = "gemini-2.5-flash"

test_row = eval_df.iloc[0]

raw_response = call_gemini_model_with_retry(
    system_prompt=test_row["system_prompt"],
    question=test_row["question"],
    model_name=TEST_MODEL_NAME,
    max_retries=3,
    base_sleep=15
)

predicted_answer = extract_option_letter(raw_response)

print("Raw response:")
print(raw_response)

print("\nExtracted answer:")
print(predicted_answer)

print("\nGold answer:")
print(test_row["answer"])

print("\nIs correct:")
print(predicted_answer == test_row["answer"])


Raw response:
C

Extracted answer:
C

Gold answer:
C

Is correct:
True


In [9]:
# -----------------------------
# Cell 9: Run debug evaluation for Gemini model
# -----------------------------
# Make sure USE_DEBUG = True in Cell 2 before running this.

MODEL_PROVIDER = "gemini"
MODEL_NAME = "gemini-2.5-flash"

debug_results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    try:
        raw_response = call_gemini_model_with_retry(
            system_prompt=row["system_prompt"],
            question=row["question"],
            model_name=MODEL_NAME,
            max_retries=3,
            base_sleep=15
        )

        predicted_answer = extract_option_letter(raw_response)

        debug_results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": raw_response,
            "predicted_answer": predicted_answer,
            "is_correct": predicted_answer == row["answer"],
            "error": None
        })

        time.sleep(1)

    except Exception as e:
        debug_results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": None,
            "predicted_answer": None,
            "is_correct": False,
            "error": str(e)
        })

debug_results_df = pd.DataFrame(debug_results)

display(debug_results_df[[
    "id",
    "task_type",
    "gold_answer",
    "predicted_answer",
    "is_correct",
    "raw_response",
    "error"
]])

valid_debug_df = debug_results_df[debug_results_df["error"].isna()].copy()

print("Debug rows:", len(debug_results_df))
print("Valid debug rows:", len(valid_debug_df))
print("Errors:", debug_results_df["error"].notna().sum())
print()
print("Debug accuracy, errors counted as wrong:", debug_results_df["is_correct"].mean())

if len(valid_debug_df) > 0:
    print("Debug accuracy, valid rows only:", valid_debug_df["is_correct"].mean())
    print()
    print("Accuracy by task, valid rows only:")
    print(valid_debug_df.groupby("task_type")["is_correct"].mean())


100%|███████████████████████████████████████████| 15/15 [02:17<00:00,  9.15s/it]


,id,task_type,gold_answer,predicted_answer,is_correct,raw_response,error
0,rs_context_v2_management_priority_0111,management_priority,C,C,True,C,None
1,rs_context_v2_management_priority_0420,management_priority,B,D,False,D,None
2,rs_context_v2_management_priority_0566,management_priority,B,D,False,D,None
3,rs_context_v2_management_priority_0078,management_priority,A,A,True,A,None
4,rs_context_v2_management_priority_0182,management_priority,D,A,False,A,None
5,rs_context_v2_pairwise_comparison_0059,pairwise_comparison,B,B,True,B,None
6,rs_context_v2_pairwise_comparison_0271,pairwise_comparison,B,A,False,A,None
7,rs_context_v2_pairwise_comparison_0515,pairwise_comparison,A,A,True,A,None
8,rs_context_v2_pairwise_comparison_0353,pairwise_comparison,A,A,True,A,None
9,rs_context_v2_pairwise_comparison_0169,pairwise_comparison,A,B,False,B,None


Debug rows: 15
Valid debug rows: 14
Errors: 1

Debug accuracy, errors counted as wrong: 0.4
Debug accuracy, valid rows only: 0.42857142857142855

Accuracy by task, valid rows only:
task_type
management_priority     0.40
pairwise_comparison     0.60
top_sensitive_region    0.25
Name: is_correct, dtype: float64


In [ ]:
# -----------------------------
# Official full evaluation: Gemini 2.5 Flash
# -----------------------------

MODEL_PROVIDER = "gemini"
MODEL_NAME = "gemini-2.5-flash"

SAVE_PATH = (
    EVAL_OUTPUT_DIR
    / "results_gemini-2.5-flash_context_v2_1800.csv"
)

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    try:
        raw_response = call_gemini_model_with_retry(
            system_prompt=row["system_prompt"],
            question=row["question"],
            model_name=MODEL_NAME,
            max_retries=8,
            base_sleep=20
        )

        predicted_answer = extract_option_letter(raw_response)

        results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "weather_condition": row.get("weather_condition"),
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": raw_response,
            "predicted_answer": predicted_answer,
            "is_correct": predicted_answer == row["answer"],
            "error": None
        })

        time.sleep(1)

    except Exception as e:
        results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "weather_condition": row.get("weather_condition"),
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": None,
            "predicted_answer": None,
            "is_correct": False,
            "error": str(e)
        })

results_df = pd.DataFrame(results)

results_df.to_csv(SAVE_PATH, index=False)

print("Saved results to:", SAVE_PATH)
print("Total saved rows:", len(results_df))
print("Number of errors:", results_df["error"].notna().sum())

print(
    "Overall accuracy including errors as wrong:",
    results_df["is_correct"].mean()
)

print("\nAccuracy by task:")
print(
    results_df
    .groupby("task_type")["is_correct"]
    .mean()
)

In [14]:
completed_df = pd.read_csv("region_sensitivity_eval_outputs/results_gemini-2.5-flash_context_v2_1800.csv")

overall_summary = (
    completed_df
    .groupby("model_name")
    .agg(
        n_questions=("id", "count"),
        accuracy=("is_correct", "mean"),
        n_errors=("error", lambda x: x.notna().sum())
    )
    .reset_index()
)

summary_by_task = (
    completed_df
    .groupby(["model_name", "task_type"])
    .agg(
        n_questions=("id", "count"),
        accuracy=("is_correct", "mean"),
        n_errors=("error", lambda x: x.notna().sum())
    )
    .reset_index()
)

display(overall_summary)
display(summary_by_task)

overall_summary.to_csv(
    "region_sensitivity_eval_outputs/overall_summary_gemini-2.5-flash_context_v2_1800.csv",
    index=False
)

summary_by_task.to_csv(
    "region_sensitivity_eval_outputs/summary_by_task_gemini-2.5-flash_context_v2_1800.csv",
    index=False
)

,model_name,n_questions,accuracy,n_errors
0,gemini-2.5-flash,1800,0.539444,0


,model_name,task_type,n_questions,accuracy,n_errors
0,gemini-2.5-flash,management_priority,600,0.511667,0
1,gemini-2.5-flash,pairwise_comparison,600,0.625000,0
2,gemini-2.5-flash,top_sensitive_region,600,0.481667,0


In [2]:
flash_path = Path("region_sensitivity_eval_outputs/results_gemini-2.5-flash_context_v2_1800.csv")

print("Flash file exists:", flash_path.exists())

if flash_path.exists():
    flash_df = pd.read_csv(flash_path)
    print("Flash rows:", len(flash_df))
    print("Flash errors:", flash_df["error"].notna().sum())
    print("Flash accuracy:", flash_df["is_correct"].mean())

Flash file exists: True
Flash rows: 1800
Flash errors: 0
Flash accuracy: 0.5394444444444444


In [ ]:
# ============================================================
# Task 5 metric verification: Gemini 2.5 Flash
# ============================================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)

# ------------------------------------------------------------
# 1. Load confirmed final result file
# ------------------------------------------------------------

file_path = (
    "region_sensitivity_eval_outputs/"
    "results_gemini-2.5-flash_context_v2_1800.csv"
)

df = pd.read_csv(file_path)

print("Selected file:")
print(file_path)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. Check required columns
# ------------------------------------------------------------

required_columns = [
    "task_type",
    "gold_answer",
    "predicted_answer",
    "is_correct",
    "error",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ------------------------------------------------------------
# 3. Standardise answers and task names
# ------------------------------------------------------------

def clean_answer(value):
    if pd.isna(value):
        return None

    answer = str(value).strip().upper()

    if answer in ["A", "B", "C", "D"]:
        return answer

    return None


df["gold_clean"] = df["gold_answer"].apply(clean_answer)
df["pred_clean"] = df["predicted_answer"].apply(clean_answer)

df["task_clean"] = (
    df["task_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

df["error_clean"] = (
    df["error"]
    .fillna("")
    .astype(str)
    .str.strip()
)

valid_df = df[
    df["gold_clean"].notna()
    & df["pred_clean"].notna()
    & (df["error_clean"] == "")
].copy()


# ------------------------------------------------------------
# 4. Basic check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC CHECK")
print("=" * 70)

print("Total rows:", len(df))
print("Valid rows:", len(valid_df))
print(
    "Invalid or empty predictions:",
    len(df) - len(valid_df)
)
print(
    "Rows with error:",
    (df["error_clean"] != "").sum()
)

print("\nTask counts in valid rows:")
print(valid_df["task_clean"].value_counts())


# ------------------------------------------------------------
# 5. Overall Accuracy and Macro-F1
# ------------------------------------------------------------

y_true = valid_df["gold_clean"]
y_pred = valid_df["pred_clean"]

overall_accuracy = accuracy_score(y_true, y_pred)

overall_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=["A", "B", "C", "D"],
    average="macro",
    zero_division=0,
)

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(f"Accuracy check: {overall_accuracy * 100:.2f}%")
print(f"Overall Macro-F1: {overall_macro_f1 * 100:.2f}%")


# ------------------------------------------------------------
# 6. Accuracy and Macro-F1 by question type
# ------------------------------------------------------------

task_order = [
    "pairwise_comparison",
    "top_sensitive_region",
    "management_priority",
]

task_rows = []

for task_name in task_order:
    sub = valid_df[
        valid_df["task_clean"] == task_name
    ].copy()

    if len(sub) == 0:
        continue

    task_accuracy = accuracy_score(
        sub["gold_clean"],
        sub["pred_clean"],
    )

    # Pairwise only has A/B.
    if task_name == "pairwise_comparison":
        task_labels = ["A", "B"]
    else:
        task_labels = ["A", "B", "C", "D"]

    task_macro_f1 = f1_score(
        sub["gold_clean"],
        sub["pred_clean"],
        labels=task_labels,
        average="macro",
        zero_division=0,
    )

    task_rows.append({
        "Task Type": task_name,
        "Valid Rows": len(sub),
        "Accuracy (%)": round(task_accuracy * 100, 2),
        "Macro-F1 (%)": round(task_macro_f1 * 100, 2),
    })

task_summary = pd.DataFrame(task_rows)

print("\n" + "=" * 70)
print("RESULTS BY QUESTION TYPE")
print("=" * 70)

display(task_summary)


# ------------------------------------------------------------
# 7. Option-level Recall and F1
# ------------------------------------------------------------

labels = ["A", "B", "C", "D"]

option_recall = recall_score(
    y_true,
    y_pred,
    labels=labels,
    average=None,
    zero_division=0,
)

option_f1 = f1_score(
    y_true,
    y_pred,
    labels=labels,
    average=None,
    zero_division=0,
)

option_summary = pd.DataFrame({
    "Option": labels,
    "Recall (%)": [
        round(value * 100, 2)
        for value in option_recall
    ],
    "F1 (%)": [
        round(value * 100, 2)
        for value in option_f1
    ],
})

print("\n" + "=" * 70)
print("OPTION-LEVEL RECALL AND F1")
print("=" * 70)

display(option_summary)


# ------------------------------------------------------------
# 8. Prediction distribution
# ------------------------------------------------------------

prediction_counts = (
    valid_df["pred_clean"]
    .value_counts()
    .reindex(labels, fill_value=0)
)

prediction_percentages = (
    prediction_counts / len(valid_df) * 100
)

prediction_summary = pd.DataFrame({
    "Option": labels,
    "Prediction Count": prediction_counts.values,
    "Prediction Percentage (%)": [
        round(value, 2)
        for value in prediction_percentages.values
    ],
})

print("\n" + "=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

display(prediction_summary)


# ------------------------------------------------------------
# 9. Consistency check against stored is_correct
# ------------------------------------------------------------

df["stored_is_correct"] = (
    df["is_correct"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

recalculated_correct = (
    df["gold_clean"] == df["pred_clean"]
)

stored_accuracy = df["stored_is_correct"].mean()
recalculated_accuracy = recalculated_correct.mean()

difference_pp = abs(
    stored_accuracy - recalculated_accuracy
) * 100

print("\n" + "=" * 70)
print("CONSISTENCY CHECK")
print("=" * 70)

print(
    f"Accuracy recalculated from answers: "
    f"{recalculated_accuracy * 100:.2f}%"
)

print(
    f"Accuracy from stored is_correct: "
    f"{stored_accuracy * 100:.2f}%"
)

print(
    f"Difference: {difference_pp:.6f} percentage points"
)